# Phase 6A - Baseline Reconciliation V0-V7

This notebook reconciles the Phase 2 baseline and the Phase 6 leakage-safe validation harness using only pre-registered variants V0-V7. It is a methodological audit, not competitive modeling.


## Boundaries

- No V8 is run.
- No HPO, model-family comparison, ensembles, submissions, public leaderboard use, or external data.
- `School` remains excluded from every feature matrix.
- `logs/experiment_log.csv` is never modified; a candidate row is written under `outputs/reports/` only.
- V1-V3 intentionally reproduce Phase 2's global preprocessing and are diagnostic-only with deliberate leakage.


In [1]:

from __future__ import annotations

from datetime import datetime
from pathlib import Path
import platform
import subprocess
import sys

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "input" / "train.csv").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing data/input/train.csv")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
EXPERIMENT_ID = "phase6a_baseline_reconciliation_v1"
PROJECT_SEED = 42
N_SPLITS = 5
ID_COL = "Id"
TARGET_COL = "Drafted"
ROLE_CATEGORICAL_COLUMNS = ["Player_Type", "Position_Type", "Position"]
PHYSICAL_IMPUTE_COLUMNS = [
    "Age",
    "Sprint_40yd",
    "Vertical_Jump",
    "Bench_Press_Reps",
    "Broad_Jump",
    "Agility_3cone",
    "Shuttle",
]
PHASE2_CV_MEAN = 0.812964
PHASE2_CV_STD_POP = 0.025740
PHASE6_FOLD_MEAN = 0.729253
PHASE6_OOF_AUC = 0.726616
BRIDGE_TOLERANCE = 0.005

DATA_DIR = PROJECT_ROOT / "data" / "input"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"
PHASE6_FOLDS_PATH = PROJECT_ROOT / "outputs" / "folds" / "phase6_rf_sanity_baseline_v1_fold_assignments.csv"
PHASE6_OOF_PATH = PROJECT_ROOT / "outputs" / "oof" / "phase6_rf_sanity_baseline_v1_oof_predictions.csv"
MAIN_LOG_PATH = PROJECT_ROOT / "logs" / "experiment_log.csv"

NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "04_phase6a_baseline_reconciliation.ipynb"
FOLD_COPY_PATH = PROJECT_ROOT / "outputs" / "folds" / "phase6a_fixed_fold_assignments.csv"
VALIDATION_SUMMARY_PATH = PROJECT_ROOT / "outputs" / "validation" / "phase6a_baseline_reconciliation_variant_summary.csv"
REPORT_PATH = PROJECT_ROOT / "outputs" / "reports" / "phase6a_baseline_reconciliation_report.md"
LOG_CANDIDATE_PATH = PROJECT_ROOT / "outputs" / "reports" / "phase6a_baseline_reconciliation_experiment_log_candidate.csv"

OOF_PATHS = {
    "phase6a_v0_phase6_current": PROJECT_ROOT / "outputs" / "oof" / "phase6a_v0_phase6_current_oof_predictions.csv",
    "phase6a_v1_phase2_replica": PROJECT_ROOT / "outputs" / "oof" / "phase6a_v1_phase2_replica_oof_predictions.csv",
    "phase6a_v2_replica_no_bmi": PROJECT_ROOT / "outputs" / "oof" / "phase6a_v2_replica_no_bmi_oof_predictions.csv",
    "phase6a_v3_replica_seed42": PROJECT_ROOT / "outputs" / "oof" / "phase6a_v3_replica_seed42_oof_predictions.csv",
    "phase6a_v4_foldsafe_ordinal_mean_bmi": PROJECT_ROOT / "outputs" / "oof" / "phase6a_v4_foldsafe_ordinal_mean_bmi_oof_predictions.csv",
    "phase6a_v5_phase6_plus_bmi": PROJECT_ROOT / "outputs" / "oof" / "phase6a_v5_phase6_plus_bmi_oof_predictions.csv",
    "phase6a_v6_phase6_ordinal": PROJECT_ROOT / "outputs" / "oof" / "phase6a_v6_phase6_ordinal_oof_predictions.csv",
    "phase6a_v7_phase6_mean_impute": PROJECT_ROOT / "outputs" / "oof" / "phase6a_v7_phase6_mean_impute_oof_predictions.csv",
}
EXPECTED_ARTIFACT_PATHS = [FOLD_COPY_PATH, VALIDATION_SUMMARY_PATH, REPORT_PATH, LOG_CANDIDATE_PATH, *OOF_PATHS.values()]

print(f"Project root: {PROJECT_ROOT}")
print(f"Experiment ID: {EXPERIMENT_ID}")
print("Phase 6A variants: V0-V7 only")

existing_artifacts = [str(path.relative_to(PROJECT_ROOT)) for path in EXPECTED_ARTIFACT_PATHS if path.exists()]
if existing_artifacts:
    raise FileExistsError("Refusing to overwrite existing Phase 6A artifacts: " + "; ".join(existing_artifacts))

missing_required = [
    str(path.relative_to(PROJECT_ROOT))
    for path in [TRAIN_PATH, TEST_PATH, SAMPLE_SUBMISSION_PATH, PHASE6_FOLDS_PATH, PHASE6_OOF_PATH, MAIN_LOG_PATH]
    if not path.exists()
]
if missing_required:
    raise FileNotFoundError("Missing required Phase 6A inputs: " + "; ".join(missing_required))

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
frozen_folds = pd.read_csv(PHASE6_FOLDS_PATH)
phase6_oof = pd.read_csv(PHASE6_OOF_PATH)
main_log_before = MAIN_LOG_PATH.read_text(encoding="utf-8")

contract_checks = []


def add_contract_check(check: str, passed: bool, notes: str = "") -> None:
    contract_checks.append({"check": check, "passed": bool(passed), "notes": notes})


add_contract_check("train_exists", TRAIN_PATH.exists(), str(TRAIN_PATH.relative_to(PROJECT_ROOT)))
add_contract_check("test_exists", TEST_PATH.exists(), str(TEST_PATH.relative_to(PROJECT_ROOT)))
add_contract_check("sample_submission_exists", SAMPLE_SUBMISSION_PATH.exists(), str(SAMPLE_SUBMISSION_PATH.relative_to(PROJECT_ROOT)))
add_contract_check("target_in_train", TARGET_COL in train.columns)
add_contract_check("target_not_in_test", TARGET_COL not in test.columns)
add_contract_check("id_in_train_test_sample", all(ID_COL in df.columns for df in [train, test, sample_submission]))
add_contract_check("sample_columns_exact", list(sample_submission.columns) == [ID_COL, TARGET_COL], str(list(sample_submission.columns)))
add_contract_check("test_sample_row_count_match", len(test) == len(sample_submission), f"test={len(test)} sample={len(sample_submission)}")
add_contract_check("test_sample_id_order_match", test[ID_COL].reset_index(drop=True).equals(sample_submission[ID_COL].reset_index(drop=True)))
add_contract_check("target_binary", set(train[TARGET_COL].dropna().astype(int).unique()) == {0, 1})
add_contract_check("train_rows_expected", len(train) == 2781, f"rows={len(train)}")
add_contract_check("test_rows_expected", len(test) == 696, f"rows={len(test)}")
contract_df = pd.DataFrame(contract_checks)
failed_contract = contract_df[~contract_df["passed"]]
if not failed_contract.empty:
    raise ValueError("Phase 6A data contract failed:\n" + failed_contract.to_string(index=False))

expected_fold_columns = [ID_COL, "fold"]
if list(frozen_folds.columns) != expected_fold_columns:
    raise ValueError(f"Frozen folds columns must be {expected_fold_columns}; got {list(frozen_folds.columns)}")
if len(frozen_folds) != len(train) or frozen_folds[ID_COL].nunique() != len(train):
    raise ValueError("Frozen folds must contain exactly one row per training Id")
if sorted(frozen_folds["fold"].unique().tolist()) != list(range(N_SPLITS)):
    raise ValueError("Frozen folds must use labels 0..4")
if not frozen_folds[ID_COL].reset_index(drop=True).equals(train[ID_COL].reset_index(drop=True)):
    raise ValueError("Frozen folds must match training Id order")

expected_oof_columns = [ID_COL, "fold", "y_true", "y_pred_proba"]
if list(phase6_oof.columns) != expected_oof_columns:
    raise ValueError(f"Phase 6 OOF columns must be {expected_oof_columns}; got {list(phase6_oof.columns)}")
if len(phase6_oof) != len(train) or phase6_oof[ID_COL].nunique() != len(train):
    raise ValueError("Phase 6 OOF must contain exactly one row per training Id")
phase6_join = phase6_oof.merge(frozen_folds, on=ID_COL, suffixes=("_oof", "_frozen"), validate="one_to_one")
if not (phase6_join["fold_oof"] == phase6_join["fold_frozen"]).all():
    raise ValueError("Phase 6 OOF fold labels must match frozen folds")
phase6_y = phase6_oof.merge(train[[ID_COL, TARGET_COL]], on=ID_COL, validate="one_to_one")
if not (phase6_y["y_true"].astype(int) == phase6_y[TARGET_COL].astype(int)).all():
    raise ValueError("Phase 6 OOF y_true must match train Drafted")

fold_array = frozen_folds["fold"].to_numpy()
y = train[TARGET_COL].astype(int).reset_index(drop=True)


def get_git_status() -> str:
    try:
        commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
        porcelain = subprocess.check_output(["git", "status", "--short"], cwd=PROJECT_ROOT, text=True).strip()
        dirty = "dirty" if porcelain else "clean"
        return f"{commit} ({dirty})"
    except Exception as exc:
        return f"git_status_unavailable: {exc}"


def format_float(value, digits: int = 6) -> str:
    if value is None or pd.isna(value):
        return ""
    return f"{float(value):.{digits}f}"


def markdown_table(df: pd.DataFrame, digits: int = 6) -> str:
    if df.empty:
        return "_No rows._"
    work = df.copy()
    columns = [str(c) for c in work.columns]

    def format_value(value):
        if pd.isna(value):
            return ""
        if isinstance(value, (float, np.floating)):
            return f"{float(value):.{digits}f}"
        return str(value).replace("\n", " ").replace("|", "\\|")

    lines = ["| " + " | ".join(columns) + " |", "|" + "|".join(["---"] * len(columns)) + "|"]
    for _, row in work.iterrows():
        lines.append("| " + " | ".join(format_value(row[col]) for col in work.columns) + " |")
    return "\n".join(lines)


def validate_probabilities(proba: np.ndarray, label: str) -> None:
    if len(proba) == 0:
        raise ValueError(f"{label}: empty probability vector")
    if pd.isna(proba).any() or not np.isfinite(proba).all():
        raise ValueError(f"{label}: probabilities contain NaN or infinite values")
    if ((proba < 0) | (proba > 1)).any():
        raise ValueError(f"{label}: probabilities outside [0, 1]")


def get_positive_class_proba(estimator, X_valid, positive_label: int = 1) -> np.ndarray:
    model = estimator.named_steps["model"] if isinstance(estimator, Pipeline) else estimator
    classes = getattr(model, "classes_", None)
    if classes is None:
        raise ValueError("Estimator classes_ unavailable after fit")
    matches = np.where(classes == positive_label)[0]
    if len(matches) != 1:
        raise ValueError(f"Positive label {positive_label} not found exactly once in classes_: {classes}")
    proba = estimator.predict_proba(X_valid)[:, int(matches[0])]
    validate_probabilities(proba, "positive_class_proba")
    return proba


def make_feature_frame(include_bmi: bool) -> pd.DataFrame:
    feature_df = train.drop(columns=[ID_COL, TARGET_COL, "School"]).copy()
    if include_bmi:
        feature_df["BMI"] = feature_df["Weight"] / (feature_df["Height"] ** 2)
    if ID_COL in feature_df.columns or TARGET_COL in feature_df.columns or "School" in feature_df.columns:
        raise ValueError("Forbidden feature leaked into feature frame")
    return feature_df


def make_rf(seed: int) -> RandomForestClassifier:
    return RandomForestClassifier(n_estimators=100, max_depth=5, random_state=seed, n_jobs=-1)


def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def summarize_oof(variant_id: str, oof: pd.DataFrame, metadata: dict) -> tuple[dict, pd.DataFrame]:
    if list(oof.columns) != expected_oof_columns:
        raise ValueError(f"{variant_id}: OOF columns must be {expected_oof_columns}")
    if len(oof) != len(train) or oof[ID_COL].nunique() != len(train):
        raise ValueError(f"{variant_id}: OOF must contain exactly one row per training Id")
    merged = oof.merge(frozen_folds, on=ID_COL, suffixes=("_oof", "_frozen"), validate="one_to_one")
    if not (merged["fold_oof"] == merged["fold_frozen"]).all():
        raise ValueError(f"{variant_id}: OOF fold labels do not match frozen folds")
    y_check = oof.merge(train[[ID_COL, TARGET_COL]], on=ID_COL, validate="one_to_one")
    if not (y_check["y_true"].astype(int) == y_check[TARGET_COL].astype(int)).all():
        raise ValueError(f"{variant_id}: y_true does not match train target")
    validate_probabilities(oof["y_pred_proba"].to_numpy(), variant_id)
    fold_rows = []
    for fold in range(N_SPLITS):
        group = oof[oof["fold"] == fold]
        if group["y_true"].nunique() < 2:
            raise ValueError(f"{variant_id}: fold {fold} has only one class")
        fold_rows.append(
            {
                "variant_id": variant_id,
                "fold": fold,
                "n": int(len(group)),
                "n_positive": int(group["y_true"].sum()),
                "n_negative": int(len(group) - group["y_true"].sum()),
                "positive_rate": float(group["y_true"].mean()),
                "roc_auc": float(roc_auc_score(group["y_true"], group["y_pred_proba"])),
            }
        )
    fold_metrics = pd.DataFrame(fold_rows)
    oof_auc = float(roc_auc_score(oof["y_true"], oof["y_pred_proba"]))
    fold_mean = float(fold_metrics["roc_auc"].mean())
    fold_std_sample = float(fold_metrics["roc_auc"].std(ddof=1))
    fold_std_population = float(fold_metrics["roc_auc"].std(ddof=0))
    summary = {
        "variant_id": variant_id,
        **metadata,
        "oof_auc": oof_auc,
        "fold_auc_mean": fold_mean,
        "fold_auc_std_sample": fold_std_sample,
        "fold_auc_std_population": fold_std_population,
        "phase2_cv_mean_delta": fold_mean - PHASE2_CV_MEAN,
        "phase6_oof_delta": oof_auc - PHASE6_OOF_AUC,
        "bridge_abs_diff_from_phase2_cv_mean": abs(fold_mean - PHASE2_CV_MEAN),
        "fold_0_auc": float(fold_metrics.loc[fold_metrics["fold"] == 0, "roc_auc"].iloc[0]),
        "fold_1_auc": float(fold_metrics.loc[fold_metrics["fold"] == 1, "roc_auc"].iloc[0]),
        "fold_2_auc": float(fold_metrics.loc[fold_metrics["fold"] == 2, "roc_auc"].iloc[0]),
        "fold_3_auc": float(fold_metrics.loc[fold_metrics["fold"] == 3, "roc_auc"].iloc[0]),
        "fold_4_auc": float(fold_metrics.loc[fold_metrics["fold"] == 4, "roc_auc"].iloc[0]),
    }
    return summary, fold_metrics


def run_global_label_variant(variant_id: str, include_bmi: bool, seed: int, metadata: dict):
    X_full = make_feature_frame(include_bmi=include_bmi)
    for col in PHYSICAL_IMPUTE_COLUMNS:
        if col in X_full.columns:
            X_full[col] = X_full[col].fillna(X_full[col].mean())
    for col in ROLE_CATEGORICAL_COLUMNS:
        if col in X_full.columns:
            encoder = LabelEncoder()
            X_full[col] = encoder.fit_transform(X_full[col].astype(str))
    if X_full.isna().any().any():
        raise ValueError(f"{variant_id}: global replica feature matrix contains NaN")
    oof_parts = []
    for fold in range(N_SPLITS):
        train_mask = fold_array != fold
        valid_mask = fold_array == fold
        model = make_rf(seed)
        model.fit(X_full.loc[train_mask], y.loc[train_mask])
        proba = get_positive_class_proba(model, X_full.loc[valid_mask], positive_label=1)
        oof_parts.append(
            pd.DataFrame(
                {
                    ID_COL: train.loc[valid_mask, ID_COL].to_numpy(),
                    "fold": fold,
                    "y_true": y.loc[valid_mask].to_numpy(),
                    "y_pred_proba": proba,
                }
            )
        )
    oof = pd.concat(oof_parts, ignore_index=True).sort_values(ID_COL).reset_index(drop=True)
    summary, fold_metrics = summarize_oof(variant_id, oof, metadata | {"feature_count": X_full.shape[1]})
    return oof, summary, fold_metrics


def run_foldsafe_variant(variant_id: str, include_bmi: bool, impute_strategy: str, encoding: str, seed: int, metadata: dict):
    X_full = make_feature_frame(include_bmi=include_bmi)
    categorical_columns = [c for c in ROLE_CATEGORICAL_COLUMNS if c in X_full.columns]
    numeric_columns = [c for c in X_full.columns if c not in categorical_columns]
    transformers = []
    if numeric_columns:
        transformers.append(("numeric", SimpleImputer(strategy=impute_strategy), numeric_columns))
    if categorical_columns:
        if encoding == "onehot":
            encoder = make_ohe()
        elif encoding == "ordinal":
            encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        else:
            raise ValueError(f"Unsupported encoding: {encoding}")
        cat_pipe = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", encoder),
            ]
        )
        transformers.append(("categorical", cat_pipe, categorical_columns))
    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
    oof_parts = []
    for fold in range(N_SPLITS):
        train_mask = fold_array != fold
        valid_mask = fold_array == fold
        pipe = Pipeline(
            [
                ("preprocessor", preprocessor),
                ("model", make_rf(seed)),
            ]
        )
        pipe.fit(X_full.loc[train_mask], y.loc[train_mask])
        proba = get_positive_class_proba(pipe, X_full.loc[valid_mask], positive_label=1)
        oof_parts.append(
            pd.DataFrame(
                {
                    ID_COL: train.loc[valid_mask, ID_COL].to_numpy(),
                    "fold": fold,
                    "y_true": y.loc[valid_mask].to_numpy(),
                    "y_pred_proba": proba,
                }
            )
        )
    oof = pd.concat(oof_parts, ignore_index=True).sort_values(ID_COL).reset_index(drop=True)
    summary, fold_metrics = summarize_oof(variant_id, oof, metadata | {"feature_count": X_full.shape[1]})
    return oof, summary, fold_metrics


def copy_phase6_oof():
    variant_id = "phase6a_v0_phase6_current"
    oof = phase6_oof.copy().sort_values(ID_COL).reset_index(drop=True)
    metadata = {
        "variant_label": "V0",
        "purpose": "Clean Phase 6 endpoint copied without retraining",
        "diagnostic_only": False,
        "deliberate_leakage": False,
        "methodologically_acceptable": True,
        "preprocessing_summary": "Phase 6 fold-safe median imputation plus role one-hot encoding; copied OOF",
        "encoding": "onehot",
        "imputation": "median",
        "include_bmi": False,
        "rf_random_state": PROJECT_SEED,
        "notes": "Copied from accepted Phase 6 OOF; no retraining in V0.",
        "feature_count": 13,
    }
    summary, fold_metrics = summarize_oof(variant_id, oof, metadata)
    return oof, summary, fold_metrics


variant_specs = [
    ("phase6a_v1_phase2_replica", "global_label", True, None, None, 2025, "V1", "Bridge validity: reproduce Phase 2 behavior under Phase 6A reporting", True, True, False, "Global train mean imputation plus global LabelEncoder plus BMI; deliberate Phase 2 leakage reproduction", "global_label_encoder", "global_mean", "Diagnostic-only; not eligible as future anchor."),
    ("phase6a_v2_replica_no_bmi", "global_label", False, None, None, 2025, "V2", "Isolate BMI contribution in the leaky Phase 2 context", True, True, False, "Global train mean imputation plus global LabelEncoder; BMI removed", "global_label_encoder", "global_mean", "Diagnostic-only; not eligible as future anchor."),
    ("phase6a_v3_replica_seed42", "global_label", True, None, None, 42, "V3", "Isolate RF random seed effect in the leaky Phase 2 context", True, True, False, "Global train mean imputation plus global LabelEncoder plus BMI; seed changed to 42", "global_label_encoder", "global_mean", "Diagnostic-only; not eligible as future anchor."),
    ("phase6a_v4_foldsafe_ordinal_mean_bmi", "foldsafe", True, "mean", "ordinal", 42, "V4", "Phase 2 recipe made fold-safe", False, False, True, "Fold-fitted mean imputation plus fold-fitted OrdinalEncoder plus BMI", "foldsafe_ordinal", "foldsafe_mean", "Acceptable construction; BMI/encoding adoption requires human decision."),
    ("phase6a_v5_phase6_plus_bmi", "foldsafe", True, "median", "onehot", 42, "V5", "Isolate BMI contribution in the clean Phase 6 context", False, False, True, "Phase 6 fold-safe median imputation plus one-hot encoding plus BMI", "foldsafe_onehot", "foldsafe_median", "BMI measured only; adoption deferred to human Block 0 decision."),
    ("phase6a_v6_phase6_ordinal", "foldsafe", False, "median", "ordinal", 42, "V6", "Isolate ordinal-vs-onehot encoding in the clean Phase 6 context", False, False, True, "Phase 6 fold-safe median imputation plus fold-fitted OrdinalEncoder", "foldsafe_ordinal", "foldsafe_median", "Decision-relevant for tree categorical encoding policy."),
    ("phase6a_v7_phase6_mean_impute", "foldsafe", False, "mean", "onehot", 42, "V7", "Isolate mean-vs-median imputation in the clean Phase 6 context", False, False, True, "Fold-fitted mean imputation plus one-hot encoding; no BMI", "foldsafe_onehot", "foldsafe_mean", "Decision-relevant for imputation statistic policy."),
]

variant_oofs = {}
summary_rows = []
fold_metric_frames = []
v0_oof, v0_summary, v0_fold_metrics = copy_phase6_oof()
variant_oofs["phase6a_v0_phase6_current"] = v0_oof
summary_rows.append(v0_summary)
fold_metric_frames.append(v0_fold_metrics)
print("V0 copied from Phase 6 OOF:", format_float(v0_summary["oof_auc"]))

for variant_id, runner, include_bmi, impute_strategy, encoding, seed, label, purpose, diagnostic, leakage, acceptable, preproc, enc_name, imp_name, notes in variant_specs:
    metadata = {
        "variant_label": label,
        "purpose": purpose,
        "diagnostic_only": diagnostic,
        "deliberate_leakage": leakage,
        "methodologically_acceptable": acceptable,
        "preprocessing_summary": preproc,
        "encoding": enc_name,
        "imputation": imp_name,
        "include_bmi": include_bmi,
        "rf_random_state": seed,
        "notes": notes,
    }
    if runner == "global_label":
        oof, summary, fold_metrics = run_global_label_variant(variant_id, include_bmi, seed, metadata)
    else:
        oof, summary, fold_metrics = run_foldsafe_variant(variant_id, include_bmi, impute_strategy, encoding, seed, metadata)
    variant_oofs[variant_id] = oof
    summary_rows.append(summary)
    fold_metric_frames.append(fold_metrics)
    print(f"{variant_id}: OOF={summary['oof_auc']:.6f}; fold_mean={summary['fold_auc_mean']:.6f}")

summary_df = pd.DataFrame(summary_rows).sort_values("variant_label").reset_index(drop=True)
fold_metrics_all = pd.concat(fold_metric_frames, ignore_index=True)
v1 = summary_df.loc[summary_df["variant_id"] == "phase6a_v1_phase2_replica"].iloc[0]
bridge_passed = bool(v1["bridge_abs_diff_from_phase2_cv_mean"] <= BRIDGE_TOLERANCE)
if not bridge_passed:
    raise RuntimeError(
        f"Bridge failure: V1 fold mean {v1['fold_auc_mean']:.6f} differs from Phase 2 {PHASE2_CV_MEAN:.6f} by {v1['bridge_abs_diff_from_phase2_cv_mean']:.6f}, tolerance {BRIDGE_TOLERANCE:.6f}."
    )

v0 = summary_df.loc[summary_df["variant_id"] == "phase6a_v0_phase6_current"].iloc[0]
if abs(v0["oof_auc"] - PHASE6_OOF_AUC) > 0.0005:
    raise RuntimeError("V0 OOF does not match accepted Phase 6 OOF anchor")

v0_by_fold = fold_metrics_all[fold_metrics_all["variant_id"] == "phase6a_v0_phase6_current"][["fold", "roc_auc"]].rename(columns={"roc_auc": "v0_roc_auc"})
paired_rows = []
for variant_id, fold_df in fold_metrics_all.groupby("variant_id"):
    merged = fold_df.merge(v0_by_fold, on="fold", how="left", validate="many_to_one")
    for row in merged.itertuples(index=False):
        paired_rows.append(
            {
                "variant_id": variant_id,
                "fold": int(row.fold),
                "variant_fold_auc": float(row.roc_auc),
                "v0_fold_auc": float(row.v0_roc_auc),
                "delta_vs_v0": float(row.roc_auc - row.v0_roc_auc),
            }
        )
paired_delta_df = pd.DataFrame(paired_rows)
paired_summary = paired_delta_df.groupby("variant_id", as_index=False).agg(
    mean_fold_delta_vs_v0=("delta_vs_v0", "mean"),
    min_fold_delta_vs_v0=("delta_vs_v0", "min"),
    max_fold_delta_vs_v0=("delta_vs_v0", "max"),
    same_sign_positive_folds=("delta_vs_v0", lambda s: int((s > 0).sum())),
    same_sign_negative_folds=("delta_vs_v0", lambda s: int((s < 0).sum())),
)
summary_df = summary_df.merge(paired_summary, on="variant_id", how="left", validate="one_to_one")
summary_df["oof_delta_vs_v0"] = summary_df["oof_auc"] - float(v0["oof_auc"])

eligible = summary_df[summary_df["methodologically_acceptable"] == True].copy()
anchor_candidate = eligible.sort_values(["oof_auc", "fold_auc_mean"], ascending=False).iloc[0]
encoding_delta = summary_df.loc[summary_df["variant_id"] == "phase6a_v6_phase6_ordinal", "oof_delta_vs_v0"].iloc[0]
bmi_clean_delta = summary_df.loc[summary_df["variant_id"] == "phase6a_v5_phase6_plus_bmi", "oof_delta_vs_v0"].iloc[0]
imputation_clean_delta = summary_df.loc[summary_df["variant_id"] == "phase6a_v7_phase6_mean_impute", "oof_delta_vs_v0"].iloc[0]
bmi_leaky_delta = summary_df.loc[summary_df["variant_id"] == "phase6a_v1_phase2_replica", "fold_auc_mean"].iloc[0] - summary_df.loc[summary_df["variant_id"] == "phase6a_v2_replica_no_bmi", "fold_auc_mean"].iloc[0]
seed_delta = summary_df.loc[summary_df["variant_id"] == "phase6a_v1_phase2_replica", "fold_auc_mean"].iloc[0] - summary_df.loc[summary_df["variant_id"] == "phase6a_v3_replica_seed42", "fold_auc_mean"].iloc[0]
global_to_foldsafe_delta = summary_df.loc[summary_df["variant_id"] == "phase6a_v3_phase2_replica", "fold_auc_mean"].iloc[0] if False else (
    summary_df.loc[summary_df["variant_id"] == "phase6a_v3_replica_seed42", "fold_auc_mean"].iloc[0]
    - summary_df.loc[summary_df["variant_id"] == "phase6a_v4_foldsafe_ordinal_mean_bmi", "fold_auc_mean"].iloc[0]
)
decomposition_df = pd.DataFrame(
    [
        {"factor": "Total Phase2 mean minus V0 OOF", "estimate": PHASE2_CV_MEAN - float(v0["oof_auc"]), "context": "Historical Phase 2 vs accepted Phase 6 OOF", "decision_use": "Gap size only"},
        {"factor": "BMI contribution, leaky context", "estimate": bmi_leaky_delta, "context": "V1 fold mean minus V2 fold mean", "decision_use": "Diagnostic-only"},
        {"factor": "RF seed effect, leaky context", "estimate": seed_delta, "context": "V1 fold mean minus V3 fold mean", "decision_use": "Diagnostic-only"},
        {"factor": "Global preprocessing effect in ordinal/mean/BMI family", "estimate": global_to_foldsafe_delta, "context": "V3 fold mean minus V4 fold mean", "decision_use": "Leakage inflation estimate"},
        {"factor": "BMI contribution, clean context", "estimate": bmi_clean_delta, "context": "V5 OOF minus V0 OOF", "decision_use": "Phase 7 Block 0 input only"},
        {"factor": "Ordinal encoding contribution, clean context", "estimate": encoding_delta, "context": "V6 OOF minus V0 OOF", "decision_use": "Encoding policy input"},
        {"factor": "Mean imputation contribution, clean context", "estimate": imputation_clean_delta, "context": "V7 OOF minus V0 OOF", "decision_use": "Imputation policy input"},
    ]
)

report_summary_cols = [
    "variant_label",
    "variant_id",
    "diagnostic_only",
    "deliberate_leakage",
    "methodologically_acceptable",
    "include_bmi",
    "encoding",
    "imputation",
    "rf_random_state",
    "feature_count",
    "oof_auc",
    "fold_auc_mean",
    "fold_auc_std_population",
    "oof_delta_vs_v0",
    "phase2_cv_mean_delta",
    "same_sign_positive_folds",
]
fold_report_cols = ["variant_id", "fold", "roc_auc"]
encoding_policy = "Fold-safe ordinal encoding is the leading tree-categorical policy candidate, pending human ratification." if encoding_delta > 0 else "One-hot remains the leading tree-categorical policy candidate, pending human ratification."
ablation_threshold = "Not confirmed yet: V0-V7 were run, but the D1 seed-sweep noise-floor diagnostic was not part of this authorized V0-V7-only execution."

report = f"""# Phase 6A Baseline Reconciliation Report

## Scope

This report reconciles the Phase 2 baseline and the Phase 6 leakage-safe validation harness using variants V0-V7 only.

Boundary checks:

- No V8 was run.
- No HPO was run.
- No submissions were generated.
- No model-family comparison was performed.
- No public leaderboard feedback was used for decisions.
- `logs/experiment_log.csv` was not modified.
- `School` was excluded from every feature matrix.

## Environment

| Item | Value |
|---|---|
| Git status | {get_git_status()} |
| Python | {sys.version.split()[0]} |
| Platform | {platform.platform()} |
| numpy | {np.__version__} |
| pandas | {pd.__version__} |
| scikit-learn | {sklearn.__version__} |

## Inputs

| Input | Path |
|---|---|
| Train | `{TRAIN_PATH.relative_to(PROJECT_ROOT)}` |
| Test | `{TEST_PATH.relative_to(PROJECT_ROOT)}` |
| Sample submission | `{SAMPLE_SUBMISSION_PATH.relative_to(PROJECT_ROOT)}` |
| Frozen folds | `{PHASE6_FOLDS_PATH.relative_to(PROJECT_ROOT)}` |
| Phase 6 OOF | `{PHASE6_OOF_PATH.relative_to(PROJECT_ROOT)}` |

## Data contract checks

{markdown_table(contract_df)}

## Bridge check

| Item | Value |
|---|---:|
| Phase 2 CV mean target | {PHASE2_CV_MEAN:.6f} |
| V1 fold mean | {float(v1['fold_auc_mean']):.6f} |
| Absolute difference | {float(v1['bridge_abs_diff_from_phase2_cv_mean']):.6f} |
| Tolerance | {BRIDGE_TOLERANCE:.6f} |
| Bridge passed | {bridge_passed} |

V1 is diagnostic-only and deliberately reproduces Phase 2 global preprocessing leakage. Passing the bridge check does not make V1 eligible as a future anchor.

## Variant summary

{markdown_table(summary_df[report_summary_cols])}

## Fold-by-fold ROC-AUC

{markdown_table(fold_metrics_all[fold_report_cols])}

## Paired fold deltas vs V0

{markdown_table(paired_delta_df)}

## Gap decomposition

{markdown_table(decomposition_df)}

## Proposed anchor candidate

The best methodologically acceptable variant in this run is:

| Field | Value |
|---|---|
| variant_id | `{anchor_candidate['variant_id']}` |
| OOF ROC-AUC | `{float(anchor_candidate['oof_auc']):.6f}` |
| fold mean ROC-AUC | `{float(anchor_candidate['fold_auc_mean']):.6f}` |
| diagnostic_only | `{anchor_candidate['diagnostic_only']}` |
| deliberate_leakage | `{anchor_candidate['deliberate_leakage']}` |

This is a recommendation for human review only. The definitive anchor must be ratified in a Phase 6A acceptance record.

## Encoding policy evidence

V6 minus V0 OOF delta: `{encoding_delta:.6f}`.

{encoding_policy}

## BMI disposition

Clean-context BMI delta, V5 minus V0 OOF: `{bmi_clean_delta:.6f}`.

BMI was measured only. Adoption remains a human Phase 7 Block 0 decision unless explicitly ratified in Phase 6A acceptance.

## Imputation evidence

Clean-context mean-imputation delta, V7 minus V0 OOF: `{imputation_clean_delta:.6f}`.

This informs imputation policy but does not authorize feature blocks, HPO, or submissions.

## Ablation threshold

{ablation_threshold}

## Unit-of-observation / D2 status

D2 was not executed in this V0-V7-only run. Unit of observation remains `Not confirmed yet`; grouped CV remains dormant unless a later approved diagnostic escalates it.

## Leakage labels

| Variants | Status |
|---|---|
| V1-V3 | Diagnostic-only; deliberate leakage by design; not eligible as future anchor. |
| V0, V4-V7 | Methodologically acceptable constructions; still require human ratification before policy adoption. |

## Artifact list

| Artifact | Path |
|---|---|
| Fixed folds copy | `{FOLD_COPY_PATH.relative_to(PROJECT_ROOT)}` |
| Variant summary | `{VALIDATION_SUMMARY_PATH.relative_to(PROJECT_ROOT)}` |
| Candidate log | `{LOG_CANDIDATE_PATH.relative_to(PROJECT_ROOT)}` |
| Report | `{REPORT_PATH.relative_to(PROJECT_ROOT)}` |

OOF artifacts:

{markdown_table(pd.DataFrame([{'variant_id': k, 'path': str(v.relative_to(PROJECT_ROOT))} for k, v in OOF_PATHS.items()]))}

## Main experiment log

`logs/experiment_log.csv` was intentionally left unchanged. Phase 6A wrote only a candidate log row under `outputs/reports/`.

## Phase 6A closure status

Phase 6A V0-V7 execution is ready for human review. Phase 6A is not formally closed until a human acceptance record ratifies the anchor, encoding policy, BMI disposition, and whether the ablation threshold remains `Not confirmed yet` or requires D1.

## Next gate

Do not start Phase 7 until Phase 6A is reviewed and accepted.
"""

candidate_log = pd.DataFrame(
    [
        {
            "experiment_id": EXPERIMENT_ID,
            "date": datetime.now().strftime("%Y-%m-%d"),
            "phase": "Phase 6A - Baseline Reconciliation",
            "notebook_or_script": str(NOTEBOOK_PATH.relative_to(PROJECT_ROOT)).replace("\\", "/"),
            "git_commit_or_status": get_git_status(),
            "data_version": "official_data_input_csvs",
            "fold_strategy": "loaded_frozen_phase6_folds; StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
            "random_seed": "42; 2025 for V1/V2 bridge replicas",
            "feature_block": "phase6a_reconciliation_v0_v7_only",
            "preprocessing_summary": "V0 copied; V1-V3 global Phase2 replicas diagnostic-only; V4-V7 fold-safe variants",
            "model_family": "RandomForestClassifier",
            "model_params_summary": "n_estimators=100; max_depth=5; random_state varies by pre-registered variant; no HPO",
            "hpo_status": "not_run",
            "cv_auc_mean": float(anchor_candidate["fold_auc_mean"]),
            "cv_auc_std": float(anchor_candidate["fold_auc_std_population"]),
            "fold_scores": "; ".join(f"{row.variant_id}:oof={row.oof_auc:.6f},fold_mean={row.fold_auc_mean:.6f}" for row in summary_df.itertuples(index=False)),
            "slice_metrics_available": False,
            "leakage_checks_passed": "acceptable_variants_pass; V1-V3 deliberate_leakage_diagnostic_only",
            "submission_created": False,
            "submission_path": None,
            "public_lb_score_if_submitted": None,
            "notes": "Phase 6A V0-V7 baseline reconciliation; no V8; no submissions; main log unchanged",
            "decision": "ready_for_phase6a_human_review",
        }
    ]
)

existing_artifacts_before_write = [str(path.relative_to(PROJECT_ROOT)) for path in EXPECTED_ARTIFACT_PATHS if path.exists()]
if existing_artifacts_before_write:
    raise FileExistsError("Refusing to overwrite existing Phase 6A artifacts before write: " + "; ".join(existing_artifacts_before_write))

for directory in [FOLD_COPY_PATH.parent, VALIDATION_SUMMARY_PATH.parent, REPORT_PATH.parent, *[path.parent for path in OOF_PATHS.values()]]:
    directory.mkdir(parents=True, exist_ok=True)

frozen_folds.to_csv(FOLD_COPY_PATH, index=False)
for variant_id, oof in variant_oofs.items():
    oof.to_csv(OOF_PATHS[variant_id], index=False)
summary_df.to_csv(VALIDATION_SUMMARY_PATH, index=False)
REPORT_PATH.write_text(report, encoding="utf-8")
candidate_log.to_csv(LOG_CANDIDATE_PATH, index=False)

main_log_after = MAIN_LOG_PATH.read_text(encoding="utf-8")
if main_log_before != main_log_after:
    raise RuntimeError("logs/experiment_log.csv changed during Phase 6A, which is forbidden")

print("Phase 6A artifacts written:")
for path in EXPECTED_ARTIFACT_PATHS:
    print(path.relative_to(PROJECT_ROOT), path.exists())
print("\nVariant summary:")
print(summary_df[["variant_label", "variant_id", "oof_auc", "fold_auc_mean", "fold_auc_std_population", "oof_delta_vs_v0", "diagnostic_only", "methodologically_acceptable"]].to_string(index=False))
print(f"\nBridge passed: {bridge_passed}; V1 fold mean={float(v1['fold_auc_mean']):.6f}; Phase2 target={PHASE2_CV_MEAN:.6f}")
print(f"Anchor candidate: {anchor_candidate['variant_id']} OOF={float(anchor_candidate['oof_auc']):.6f}")


Project root: C:\GitHub\reto_Tokio
Experiment ID: phase6a_baseline_reconciliation_v1
Phase 6A variants: V0-V7 only
V0 copied from Phase 6 OOF: 0.726616


phase6a_v1_phase2_replica: OOF=0.811638; fold_mean=0.812964


phase6a_v2_replica_no_bmi: OOF=0.810277; fold_mean=0.811531


phase6a_v3_replica_seed42: OOF=0.808729; fold_mean=0.810717


phase6a_v4_foldsafe_ordinal_mean_bmi: OOF=0.809010; fold_mean=0.812399


phase6a_v5_phase6_plus_bmi: OOF=0.726948; fold_mean=0.729105


phase6a_v6_phase6_ordinal: OOF=0.725812; fold_mean=0.728038


phase6a_v7_phase6_mean_impute: OOF=0.802271; fold_mean=0.805851


Phase 6A artifacts written:
outputs\folds\phase6a_fixed_fold_assignments.csv True
outputs\validation\phase6a_baseline_reconciliation_variant_summary.csv True
outputs\reports\phase6a_baseline_reconciliation_report.md True
outputs\reports\phase6a_baseline_reconciliation_experiment_log_candidate.csv True
outputs\oof\phase6a_v0_phase6_current_oof_predictions.csv True
outputs\oof\phase6a_v1_phase2_replica_oof_predictions.csv True
outputs\oof\phase6a_v2_replica_no_bmi_oof_predictions.csv True
outputs\oof\phase6a_v3_replica_seed42_oof_predictions.csv True
outputs\oof\phase6a_v4_foldsafe_ordinal_mean_bmi_oof_predictions.csv True
outputs\oof\phase6a_v5_phase6_plus_bmi_oof_predictions.csv True
outputs\oof\phase6a_v6_phase6_ordinal_oof_predictions.csv True
outputs\oof\phase6a_v7_phase6_mean_impute_oof_predictions.csv True

Variant summary:
variant_label                           variant_id  oof_auc  fold_auc_mean  fold_auc_std_population  oof_delta_vs_v0  diagnostic_only  methodologically_accepta